# AHMET ÇALIK VAKFI – CASE STUDY 02  
## Notebook 01: Veri Keşfi (EDA) ve Veri Temizliği

Bu notebook aşağıdaki adımları içerir:

- Veri setlerinin okunması ve sütun adlarının standartlaştırılması
- Veri tiplerinin dönüştürülmesi
- Genel veri yapısının incelenmesi
- Eksik veri, mükerrer kayıt, negatif ve sıfır tüketim analizi
- İlçe ve hesap sınıfı bazında tanımlayıcı istatistikler
- Hesap sınıfı ve ilçe bazında IQR yöntemiyle aykırı değer işaretleme
- Müşteri bazında yüksek tüketimli müşteri analizi
- Temizlenmiş ve işaretlenmiş çıktıların kaydedilmesi

> Not: Negatif tüketimler ve aykırı değerler otomatik olarak “hata” veya “VIP” kabul edilmez. Önce ayrı olarak işaretlenir ve analiz edilir.


In [2]:
# ======================================================================
# AHMET ÇALIK VAKFI - CASE STUDY 02
# NOTEBOOK 01 : VERİ KEŞFİ VE VERİ TEMİZLİĞİ
# ======================================================================

import os
import re
import warnings
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

print("=" * 80)
print("NOTEBOOK 01 - VERİ KEŞFİ (EDA) VE VERİ KALİTESİ")
print("=" * 80)

# ======================================================================
# 1. DOSYA YOLLARI VE ÇIKTI KLASÖRÜ
# ======================================================================

DOSYA_YOLLARI = {
    "tahsilat": r"C:\veri_1.csv",
    "tahsilat_ozet": r"C:\veri_2.csv",
    "tahakkuk_hamamozu": r"C:\veri_3.csv",
    "tahakkuk_gumushacikoy": r"C:\veri_4.csv",
    "tahakkuk_goynucek": r"C:\veri_5.csv",
}

CIKTI_KLASORU = os.path.join(os.getcwd(), "notebook_01_ciktilari")
os.makedirs(CIKTI_KLASORU, exist_ok=True)

print(f"\nÇıktı klasörü: {CIKTI_KLASORU}")



NOTEBOOK 01 - VERİ KEŞFİ (EDA) VE VERİ KALİTESİ

Çıktı klasörü: C:\Users\ÇaY GüZeLi\Downloads\notebook_01_ciktilari


In [ ]:

# ======================================================================
# 2. YARDIMCI FONKSİYONLAR
# ======================================================================

def oku(path):
    """Noktalı virgül ayrımlı ve ondalık virgüllü CSV dosyasını okur."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"Dosya bulunamadı: {path}")

    return pd.read_csv(
        path,
        sep=";",
        decimal=",",
        encoding="utf-8-sig",
        low_memory=False
    )


def turkce_karakterleri_duzelt(metin):
    """Türkçe karakterleri ASCII karşılıklarına dönüştürür."""
    ceviri = str.maketrans({
        "ı": "i", "İ": "i",
        "ğ": "g", "Ğ": "g",
        "ü": "u", "Ü": "u",
        "ş": "s", "Ş": "s",
        "ö": "o", "Ö": "o",
        "ç": "c", "Ç": "c"
    })
    return str(metin).translate(ceviri)


def temiz_sutun_adi(sutun):
    """Sütun adlarını küçük harfli, Türkçe karaktersiz ve alt çizgili forma getirir."""
    sutun = str(sutun).replace("\ufeff", "").strip().casefold()
    sutun = turkce_karakterleri_duzelt(sutun)

    # Özellikle büyük İ harfinin casefold sonrası oluşturabildiği birleşik noktayı temizler.
    sutun = unicodedata.normalize("NFKD", sutun)
    sutun = "".join(
        karakter for karakter in sutun
        if not unicodedata.combining(karakter)
    )

    sutun = re.sub(r"[^a-z0-9]+", "_", sutun)
    sutun = re.sub(r"_+", "_", sutun).strip("_")
    return sutun


def standartlastir(df):
    """
    Sütun adlarını standartlaştırır.

    Tahakkuk tablolarında bulunan iki farklı alanı özellikle ayırır:
    - hesap_sinifi_kodu: M001 gibi sınıf kodları
    - hesap_sinifi: Mesken, Ticari vb. açıklamalar

    Böylece aynı isimli iki sütun oluşması ve
    'Grouper ... not 1-dimensional' hatası engellenir.
    """
    df = df.copy()

    orijinal_sutunlar = list(df.columns)
    standart_adlar = [temiz_sutun_adi(col) for col in orijinal_sutunlar]

    # Aynı normalize edilmiş ad birden fazla kez oluşursa geçici benzersiz ad ver.
    sayac = {}
    benzersiz_adlar = []

    for ad in standart_adlar:
        sira = sayac.get(ad, 0)
        benzersiz_adlar.append(ad if sira == 0 else f"{ad}__{sira}")
        sayac[ad] = sira + 1

    df.columns = benzersiz_adlar

    # Hesap sınıfının kod ve açıklama sütunlarını birbirinden ayır.
    hesap_adaylari = [
        col for col in df.columns
        if col == "hesap_sinifi" or col.startswith("hesap_sinifi__")
    ]

    if len(hesap_adaylari) >= 2:
        kod_sutunu = None
        aciklama_sutunu = None

        for col in hesap_adaylari:
            ornek = df[col].dropna().astype(str).str.strip().head(1000)

            # M001, T003 vb. kod formatına benzeyen değerlerin oranı
            kod_orani = (
                ornek.str.match(r"^[A-Za-z]+\d+$", na=False).mean()
                if len(ornek) > 0 else 0
            )

            if kod_orani >= 0.50 and kod_sutunu is None:
                kod_sutunu = col
            elif aciklama_sutunu is None:
                aciklama_sutunu = col

        if kod_sutunu is None:
            kod_sutunu = hesap_adaylari[0]

        if aciklama_sutunu is None:
            aciklama_sutunu = next(
                (col for col in hesap_adaylari if col != kod_sutunu),
                None
            )

        yeniden_adlandir = {kod_sutunu: "hesap_sinifi_kodu"}

        if aciklama_sutunu is not None:
            yeniden_adlandir[aciklama_sutunu] = "hesap_sinifi"

        df = df.rename(columns=yeniden_adlandir)

        # İkiden fazla eş adlı sütun varsa kalanları boşluk tamamlamada kullan.
        kalan_hesap_sutunlari = [
            col for col in df.columns
            if col.startswith("hesap_sinifi__")
        ]

        for col in kalan_hesap_sutunlari:
            if "hesap_sinifi" in df.columns:
                df["hesap_sinifi"] = df["hesap_sinifi"].fillna(df[col])
            df = df.drop(columns=col)

    elif len(hesap_adaylari) == 1:
        df = df.rename(columns={hesap_adaylari[0]: "hesap_sinifi"})

    # Eş anlamlı müşteri ve dönem sütunlarını standart adlara getir.
    yeniden_adlandirma = {}

    for col in df.columns:
        if col in {
            "soz_hsp_bagimsiz",
            "soz_hsp_bagimsiz_",
            "sozlesme_hesabi",
            "sozlesme_hesap"
        }:
            yeniden_adlandirma[col] = "sozlesme_hesap_no"
        elif col == "mali_yil_donemi":
            yeniden_adlandirma[col] = "mali_yil_donem"
        elif col == "tuketim_kwh":
            yeniden_adlandirma[col] = "kwh"
        elif col in {"i_lce", "i_lce_"}:
            yeniden_adlandirma[col] = "ilce"
        elif col in {"i_l", "i_l_"}:
            yeniden_adlandirma[col] = "il"

    df = df.rename(columns=yeniden_adlandirma)

    # Yeniden adlandırma sonrasında oluşabilecek gerçek eş anlamlı tekrarları birleştir.
    for hedef in ["sozlesme_hesap_no", "mali_yil_donem", "il", "ilce", "kwh"]:
        ayni_adli = [i for i, col in enumerate(df.columns) if col == hedef]

        if len(ayni_adli) > 1:
            birlesik = df.iloc[:, ayni_adli].bfill(axis=1).iloc[:, 0]
            tutulacaklar = [
                i for i in range(df.shape[1])
                if i not in ayni_adli
            ]
            df = df.iloc[:, tutulacaklar]
            df[hedef] = birlesik

    if df.columns.duplicated().any():
        tekrarlar = df.columns[df.columns.duplicated()].tolist()
        raise ValueError(
            f"Standartlaştırma sonrasında yinelenen sütun adları kaldı: {tekrarlar}"
        )

    return df


def tarih_donustur(df, sutunlar):
    """Belirtilen sütunları güvenli biçimde datetime tipine dönüştürür."""
    df = df.copy()

    for col in sutunlar:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    return df


def musteri_id_duzelt(df):
    """Müşteri numaralarını metin tipine dönüştürür."""
    df = df.copy()

    if "sozlesme_hesap_no" in df.columns:
        df["sozlesme_hesap_no"] = (
            df["sozlesme_hesap_no"]
            .astype("string")
            .str.replace(r"\.0$", "", regex=True)
            .str.strip()
        )

    return df


def eksik_deger_ozeti(df):
    """Eksik değer sayısı ve oranını döndürür."""
    ozet = pd.DataFrame({
        "eksik_sayi": df.isna().sum(),
        "eksik_oran_yuzde": df.isna().mean() * 100
    })

    return ozet.sort_values(
        ["eksik_oran_yuzde", "eksik_sayi"],
        ascending=False
    )


def guvenli_oran(pay, payda):
    """Sıfıra bölme hatasını önleyerek yüzde hesaplar."""
    if payda == 0:
        return np.nan
    return (pay / payda) * 100


def mod_deger(series):
    """Bir serinin en sık görülen değerini güvenli biçimde döndürür."""
    mod = series.dropna().mode()
    return mod.iloc[0] if not mod.empty else np.nan




In [3]:
# ======================================================================
# 3. DOSYALARI OKU
# ======================================================================

print("\n[ADIM 1] Dosyalar okunuyor...")

df_tahsilat = standartlastir(oku(DOSYA_YOLLARI["tahsilat"]))
df_tahsilat_ozet = standartlastir(oku(DOSYA_YOLLARI["tahsilat_ozet"]))

df_hamamozu = standartlastir(oku(DOSYA_YOLLARI["tahakkuk_hamamozu"]))
df_gumushacikoy = standartlastir(oku(DOSYA_YOLLARI["tahakkuk_gumushacikoy"]))
df_goynucek = standartlastir(oku(DOSYA_YOLLARI["tahakkuk_goynucek"]))

# Kaynak sayfayı izleyebilmek için ilçe bilgisi ekle
df_hamamozu["ilce_kaynak"] = "Hamamözü"
df_gumushacikoy["ilce_kaynak"] = "Gümüşhacıköy"
df_goynucek["ilce_kaynak"] = "Göynücek"

df = pd.concat(
    [df_hamamozu, df_gumushacikoy, df_goynucek],
    ignore_index=True
)

# Yinelenen sütun adı güvenlik kontrolü
if df.columns.duplicated().any():
    tekrar_eden_sutunlar = df.columns[df.columns.duplicated()].tolist()
    raise ValueError(
        f"Tahakkuk verisinde yinelenen sütun adları bulundu: {tekrar_eden_sutunlar}"
    )

print(" Dosyalar başarıyla okundu.")
print(f"Tahsilat satır sayısı       : {len(df_tahsilat):,}")
print(f"Tahsilat özet satır sayısı  : {len(df_tahsilat_ozet):,}")
print(f"Tahakkuk toplam satır sayısı: {len(df):,}")


# ======================================================================
# 4. VERİ TİPİ DÖNÜŞÜMLERİ
# ======================================================================

print("\n[ADIM 2] Veri tipleri dönüştürülüyor...")

# kWh
if "kwh" not in df.columns:
    raise KeyError("Tahakkuk verisinde 'kwh' sütunu bulunamadı.")

df["kwh"] = pd.to_numeric(df["kwh"], errors="coerce")

# Tarihler
df = tarih_donustur(
    df,
    ["mali_yil_donem", "fatura_tarihi", "kayit_tarihi", "vade_tarihi"]
)

df_tahsilat = tarih_donustur(
    df_tahsilat,
    ["tahsilat_tarihi", "odeme_tarihi", "fatura_tarihi", "vade_tarihi"]
)

df_tahsilat_ozet = tarih_donustur(
    df_tahsilat_ozet,
    ["mali_yil_donem", "fatura_tarihi", "vade_tarihi", "tahsilat_tarihi"]
)

# Müşteri numaraları
df = musteri_id_duzelt(df)
df_tahsilat = musteri_id_duzelt(df_tahsilat)
df_tahsilat_ozet = musteri_id_duzelt(df_tahsilat_ozet)

# İlçe sütunu boşsa kaynak sayfa bilgisini kullan
if "ilce" not in df.columns:
    df["ilce"] = df["ilce_kaynak"]
else:
    df["ilce"] = df["ilce"].fillna(df["ilce_kaynak"])

print(" Veri tipi dönüşümleri tamamlandı.")





[ADIM 1] Dosyalar okunuyor...
✓ Dosyalar başarıyla okundu.
Tahsilat satır sayısı       : 636,993
Tahsilat özet satır sayısı  : 917,632
Tahakkuk toplam satır sayısı: 1,185,698

[ADIM 2] Veri tipleri dönüştürülüyor...
✓ Veri tipi dönüşümleri tamamlandı.


In [4]:
# ======================================================================
# 5. GENEL VERİ KEŞFİ
# ======================================================================

print("\n" + "=" * 80)
print("GENEL VERİ KEŞFİ")
print("=" * 80)

veri_setleri = {
    "Tahsilat": df_tahsilat,
    "Tahsilat Özet": df_tahsilat_ozet,
    "Tahakkuk": df
}

genel_ozet_satirlari = []

for ad, veri in veri_setleri.items():
    print(f"\n{'-' * 80}")
    print(ad.upper())
    print(f"{'-' * 80}")
    print(f"Boyut: {veri.shape[0]:,} satır x {veri.shape[1]:,} sütun")

    display(veri.head())

    print("\nVeri tipleri:")
    display(veri.dtypes.to_frame("veri_tipi"))

    genel_ozet_satirlari.append({
        "veri_seti": ad,
        "satir_sayisi": veri.shape[0],
        "sutun_sayisi": veri.shape[1],
        "tam_mukerrer_satir": veri.duplicated().sum(),
        "toplam_eksik_hucre": veri.isna().sum().sum()
    })

genel_ozet = pd.DataFrame(genel_ozet_satirlari)
display(genel_ozet)

genel_ozet.to_csv(
    os.path.join(CIKTI_KLASORU, "genel_veri_seti_ozeti.csv"),
    sep=";",
    decimal=",",
    encoding="utf-8-sig",
    index=False
)


# ======================================================================
# 6. TEMEL TAHakkUK METRİKLERİ
# ======================================================================

print("\n" + "=" * 80)
print("TAHAKKUK TEMEL METRİKLERİ")
print("=" * 80)

toplam_kayit = len(df)
tekil_musteri = (
    df["sozlesme_hesap_no"].nunique(dropna=True)
    if "sozlesme_hesap_no" in df.columns
    else np.nan
)

gecerli_kwh = df["kwh"].dropna()

ortalama_kwh = gecerli_kwh.mean()
medyan_kwh = gecerli_kwh.median()
std_kwh = gecerli_kwh.std()
min_kwh = gecerli_kwh.min()
maks_kwh = gecerli_kwh.max()
q1_kwh = gecerli_kwh.quantile(0.25)
q3_kwh = gecerli_kwh.quantile(0.75)

print(f"Tahakkuk kayıt sayısı       : {toplam_kayit:,}")
print(f"Tekil müşteri sayısı        : {tekil_musteri:,.0f}")
print(f"Geçerli kWh kayıt sayısı    : {gecerli_kwh.shape[0]:,}")
print(f"Ortalama tüketim            : {ortalama_kwh:,.2f} kWh")
print(f"Medyan tüketim              : {medyan_kwh:,.2f} kWh")
print(f"Standart sapma              : {std_kwh:,.2f} kWh")
print(f"Minimum tüketim             : {min_kwh:,.2f} kWh")
print(f"Birinci çeyrek (Q1)         : {q1_kwh:,.2f} kWh")
print(f"Üçüncü çeyrek (Q3)          : {q3_kwh:,.2f} kWh")
print(f"Maksimum tüketim            : {maks_kwh:,.2f} kWh")

if pd.notna(ortalama_kwh) and pd.notna(medyan_kwh):
    if ortalama_kwh > medyan_kwh:
        print(
            "\nYorum: Ortalama tüketimin medyandan yüksek olması, dağılımın sağa çarpık "
            "olduğunu ve yüksek tüketimli az sayıdaki kaydın ortalamayı yukarı çektiğini gösterir."
        )
    elif ortalama_kwh < medyan_kwh:
        print(
            "\nYorum: Ortalama tüketimin medyandan düşük olması, dağılımın sola çarpık "
            "olabileceğine işaret eder."
        )
    else:
        print("\nYorum: Ortalama ve medyanın eşit olması dağılımın görece simetrik olabileceğine işaret eder.")




GENEL VERİ KEŞFİ

--------------------------------------------------------------------------------
TAHSILAT
--------------------------------------------------------------------------------
Boyut: 636,993 satır x 9 sütun


,sube,kasa,ilce,sozlesme_hesap_no,tahsilat_tarihi,nakit_tahsilat,mahsuben_tahsilat,kredi_karti_tahsilati,banka_tahsilati
0,Tayin edilmedi,Tayin edilmedi,TAŞOVA,4989745446,2023-11-06,NaN,"8,648.95",NaN,NaN
1,Tayin edilmedi,Tayin edilmedi,TAŞOVA,4989745446,2024-06-26,NaN,762.40,NaN,NaN
2,Tayin edilmedi,Tayin edilmedi,TAŞOVA,4989745446,2024-07-10,NaN,311.60,NaN,NaN
3,PTT,PTT/PV,TAŞOVA,4254955886,2023-01-19,NaN,NaN,NaN,130.50
4,PTT,PTT/PV,TAŞOVA,4254955886,2023-02-17,NaN,NaN,NaN,117.00



Veri tipleri:


,veri_tipi
sube,object
kasa,object
ilce,object
sozlesme_hesap_no,string[python]
tahsilat_tarihi,datetime64[ns]
nakit_tahsilat,float64
mahsuben_tahsilat,float64
kredi_karti_tahsilati,float64
banka_tahsilati,float64



--------------------------------------------------------------------------------
TAHSILAT ÖZET
--------------------------------------------------------------------------------
Boyut: 917,632 satır x 22 sütun


...

,il,ilce,sozlesme_hesap_no,mali_yil_donem,fatura_tarihi,kayit_tarihi,vade_tarihi,hesap_sinifi_kodu,hesap_sinifi,kwh,ilce_kaynak
0,AMASYA,HAMAMÖZÜ,917576806,2023-01-01,2023-01-12,2023-03-06,2023-01-23,M001,Mesken,1.79,Hamamözü
1,AMASYA,HAMAMÖZÜ,917576806,2023-01-01,2023-02-09,2023-05-11,2023-02-20,M001,Mesken,2.60,Hamamözü
2,AMASYA,HAMAMÖZÜ,917576806,2023-02-01,2023-02-09,2023-05-11,2023-02-20,M001,Mesken,1.23,Hamamözü
3,AMASYA,HAMAMÖZÜ,917576806,2023-02-01,2023-03-10,2023-05-11,2023-03-20,M001,Mesken,2.56,Hamamözü
4,AMASYA,HAMAMÖZÜ,917576806,2023-03-01,2023-03-10,2023-05-11,2023-03-20,M001,Mesken,1.35,Hamamözü



Veri tipleri:


,veri_tipi
il,object
ilce,object
sozlesme_hesap_no,string[python]
mali_yil_donem,datetime64[ns]
fatura_tarihi,datetime64[ns]
kayit_tarihi,datetime64[ns]
vade_tarihi,datetime64[ns]
hesap_sinifi_kodu,object
hesap_sinifi,object
kwh,float64


,veri_seti,satir_sayisi,sutun_sayisi,tam_mukerrer_satir,toplam_eksik_hucre
0,Tahsilat,636993,9,0,1910974
1,Tahsilat Özet,917632,22,246,14211283
2,Tahakkuk,1185698,11,0,0



TAHAKKUK TEMEL METRİKLERİ
Tahakkuk kayıt sayısı       : 1,185,698
Tekil müşteri sayısı        : 28,290
Geçerli kWh kayıt sayısı    : 1,185,698
Ortalama tüketim            : 92.64 kWh
Medyan tüketim              : 46.62 kWh
Standart sapma              : 950.41 kWh
Minimum tüketim             : -25,370.64 kWh
Birinci çeyrek (Q1)         : 18.01 kWh
Üçüncü çeyrek (Q3)          : 80.00 kWh
Maksimum tüketim            : 153,575.73 kWh

Yorum: Ortalama tüketimin medyandan yüksek olması, dağılımın sağa çarpık olduğunu ve yüksek tüketimli az sayıdaki kaydın ortalamayı yukarı çektiğini gösterir.


In [5]:

# ======================================================================
# 7. EKSİK VERİ ANALİZİ
# ======================================================================

print("\n" + "=" * 80)
print("EKSİK VERİ ANALİZİ")
print("=" * 80)

eksik_raporlari = {}

for ad, veri in veri_setleri.items():
    ozet = eksik_deger_ozeti(veri)
    eksik_raporlari[ad] = ozet

    print(f"\n{ad} veri seti eksik değer özeti:")
    eksik_var = ozet[ozet["eksik_sayi"] > 0]

    if eksik_var.empty:
        print("Eksik veri bulunmamaktadır.")
    else:
        display(eksik_var)

        dosya_adi = (
            ad.lower()
            .replace(" ", "_")
            .replace("ö", "o")
            .replace("ı", "i")
        )

        eksik_var.to_csv(
            os.path.join(CIKTI_KLASORU, f"eksik_deger_ozeti_{dosya_adi}.csv"),
            sep=";",
            decimal=",",
            encoding="utf-8-sig"
        )

# Tahakkuk kWh eksiklerinin ilçe bazında dağılımı
if "ilce" in df.columns:
    eksik_kwh_ilce = (
        df.assign(kwh_eksik=df["kwh"].isna())
        .groupby("ilce", dropna=False)
        .agg(
            toplam_kayit=("kwh_eksik", "size"),
            eksik_kwh=("kwh_eksik", "sum"),
            eksik_kwh_orani=("kwh_eksik", "mean")
        )
        .reset_index()
    )

    eksik_kwh_ilce["eksik_kwh_orani"] *= 100

    print("\nkWh eksiklerinin ilçe bazında dağılımı:")
    display(eksik_kwh_ilce)

    eksik_kwh_ilce.to_csv(
        os.path.join(CIKTI_KLASORU, "eksik_kwh_ilce_ozeti.csv"),
        sep=";",
        decimal=",",
        encoding="utf-8-sig",
        index=False
    )

print(
    "\nNot: Eksik bir tarih veya ödeme alanı, veri sözlüğü ile doğrulanmadan doğrudan "
    "'ödenmemiş fatura' veya 'riskli müşteri' olarak yorumlanmamıştır."
)


# ======================================================================
# 8. MÜKERRER KAYIT ANALİZİ
# ======================================================================

print("\n" + "=" * 80)
print("MÜKERRER KAYIT ANALİZİ")
print("=" * 80)

tam_mukerrer_sayisi = df.duplicated().sum()
print(f"Tamamen aynı olan mükerrer satır sayısı: {tam_mukerrer_sayisi:,}")

is_anahtari = [
    col for col in
    ["sozlesme_hesap_no", "mali_yil_donem", "fatura_tarihi"]
    if col in df.columns
]

if len(is_anahtari) >= 2:
    is_anahtari_mukerrer = df.duplicated(
        subset=is_anahtari,
        keep=False
    )

    df_is_anahtari_mukerrer = df[is_anahtari_mukerrer].copy()

    print(
        f"{' + '.join(is_anahtari)} bazında tekrar eden satır sayısı: "
        f"{len(df_is_anahtari_mukerrer):,}"
    )

    df_is_anahtari_mukerrer.to_csv(
        os.path.join(CIKTI_KLASORU, "is_anahtari_bazli_mukerrer_kayitlar.csv"),
        sep=";",
        decimal=",",
        encoding="utf-8-sig",
        index=False
    )

print(
    "\nNot: Aynı müşteri ve dönemde birden fazla kayıt bulunması otomatik olarak hata kabul edilmemiştir. "
    "Bu kayıtlar fatura düzeltmesi, iptal veya yeniden tahakkuk işlemi olabilir."
)


# ======================================================================
# 9. NEGATİF, SIFIR VE POZİTİF TÜKETİM ANALİZİ
# ======================================================================

print("\n" + "=" * 80)
print("NEGATİF, SIFIR VE POZİTİF TÜKETİM ANALİZİ")
print("=" * 80)

df_negatif = df[df["kwh"] < 0].copy()
df_sifir = df[df["kwh"] == 0].copy()
df_pozitif = df[df["kwh"] > 0].copy()
df_kwh_eksik = df[df["kwh"].isna()].copy()

tuketim_durum_ozeti = pd.DataFrame({
    "durum": ["Eksik", "Negatif", "Sıfır", "Pozitif"],
    "kayit_sayisi": [
        len(df_kwh_eksik),
        len(df_negatif),
        len(df_sifir),
        len(df_pozitif)
    ]
})

tuketim_durum_ozeti["oran_yuzde"] = (
    tuketim_durum_ozeti["kayit_sayisi"] / len(df) * 100
)

display(tuketim_durum_ozeti)

print(
    "\nYorum: Negatif tüketimler doğrudan veri hatası olarak kabul edilmemiştir. "
    "İptal, düzeltme, mahsuplaşma veya veri aktarım süreçlerinden kaynaklanabilir. "
    "Bu nedenle ana pozitif tüketim analizinden ayrı tutulmuş ve ayrıca dışarı aktarılmıştır."
)

# Negatif tüketimlerin ilçe ve hesap sınıfı özeti
negatif_gruplama = [
    col for col in ["ilce", "hesap_sinifi"]
    if col in df_negatif.columns
]

if not df_negatif.empty and negatif_gruplama:
    negatif_ozet = (
        df_negatif.groupby(negatif_gruplama, dropna=False)
        .agg(
            negatif_kayit_sayisi=("kwh", "size"),
            toplam_negatif_kwh=("kwh", "sum"),
            ortalama_negatif_kwh=("kwh", "mean"),
            minimum_kwh=("kwh", "min")
        )
        .reset_index()
        .sort_values("negatif_kayit_sayisi", ascending=False)
    )

    display(negatif_ozet.head(20))

    negatif_ozet.to_csv(
        os.path.join(CIKTI_KLASORU, "negatif_tuketim_grup_ozeti.csv"),
        sep=";",
        decimal=",",
        encoding="utf-8-sig",
        index=False
    )





EKSİK VERİ ANALİZİ

Tahsilat veri seti eksik değer özeti:


,eksik_sayi,eksik_oran_yuzde
kredi_karti_tahsilati,636993,100.00
nakit_tahsilat,636470,99.92
mahsuben_tahsilat,629451,98.82
banka_tahsilati,8060,1.27



Tahsilat Özet veri seti eksik değer özeti:


,eksik_sayi,eksik_oran_yuzde
son_odeme_150_180,916081,99.83
son_odeme_120_150,915330,99.75
son_odeme_90_120,913812,99.58
son_odeme_180,912805,99.47
son_odeme_60_90,910448,99.22
son_odeme_5,910309,99.20
son_odeme_4,900637,98.15
son_odeme_3,898739,97.94
son_odeme_1,896730,97.72
son_odeme_2,895968,97.64



Tahakkuk veri seti eksik değer özeti:
Eksik veri bulunmamaktadır.

kWh eksiklerinin ilçe bazında dağılımı:


,ilce,toplam_kayit,eksik_kwh,eksik_kwh_orani
0,GÖYNÜCEK,295223,0,0.00
1,GÜMÜŞHACIKÖY,765657,0,0.00
2,HAMAMÖZÜ,124818,0,0.00



Not: Eksik bir tarih veya ödeme alanı, veri sözlüğü ile doğrulanmadan doğrudan 'ödenmemiş fatura' veya 'riskli müşteri' olarak yorumlanmamıştır.

MÜKERRER KAYIT ANALİZİ
Tamamen aynı olan mükerrer satır sayısı: 0
sozlesme_hesap_no + mali_yil_donem + fatura_tarihi bazında tekrar eden satır sayısı: 210

Not: Aynı müşteri ve dönemde birden fazla kayıt bulunması otomatik olarak hata kabul edilmemiştir. Bu kayıtlar fatura düzeltmesi, iptal veya yeniden tahakkuk işlemi olabilir.

NEGATİF, SIFIR VE POZİTİF TÜKETİM ANALİZİ


,durum,kayit_sayisi,oran_yuzde
0,Eksik,0,0.00
1,Negatif,151,0.01
2,Sıfır,55377,4.67
3,Pozitif,1130170,95.32



Yorum: Negatif tüketimler doğrudan veri hatası olarak kabul edilmemiştir. İptal, düzeltme, mahsuplaşma veya veri aktarım süreçlerinden kaynaklanabilir. Bu nedenle ana pozitif tüketim analizinden ayrı tutulmuş ve ayrıca dışarı aktarılmıştır.


,ilce,hesap_sinifi,negatif_kayit_sayisi,toplam_negatif_kwh,ortalama_negatif_kwh,minimum_kwh
6,GÜMÜŞHACIKÖY,Mesken,79,"-22,018.98",-278.72,"-1,686.44"
0,GÖYNÜCEK,Mesken,23,"-5,125.96",-222.87,"-2,836.64"
9,GÜMÜŞHACIKÖY,Ticari Faaliyet - Yazıhane,13,"-15,103.84","-1,161.83","-3,861.20"
3,GÖYNÜCEK,Tarımsal Faaliyetler (Kooperatif),10,"-6,775.72",-677.57,"-4,208.64"
7,GÜMÜŞHACIKÖY,Tarımsal Faaliyetler (Kooperatif),6,"-14,737.68","-2,456.28","-12,574.78"
8,GÜMÜŞHACIKÖY,Tarımsal Faaliyetler (Şahıs),6,"-16,382.20","-2,730.37","-9,069.13"
4,GÖYNÜCEK,Tarımsal Faaliyetler (Şahıs),4,-494.88,-123.72,-278.16
11,HAMAMÖZÜ,Mesken,4,"-1,978.41",-494.60,"-1,242.99"
1,GÖYNÜCEK,Resmi Daire,2,"-2,556.00","-1,278.00","-1,484.13"
10,GÜMÜŞHACIKÖY,Şantiye ve Geçici Aboneler,2,-479.82,-239.91,-298.82


In [6]:
# ======================================================================
# 10. İLÇE BAZINDA TANIMLAYICI İSTATİSTİKLER
# ======================================================================

print("\n" + "=" * 80)
print("İLÇE BAZINDA TANIMLAYICI İSTATİSTİKLER")
print("=" * 80)

ilce_agg = {
    "kayit_sayisi": ("kwh", "size"),
    "gecerli_kwh_sayisi": ("kwh", "count"),
    "toplam_tuketim_kwh": ("kwh", "sum"),
    "ortalama_kwh": ("kwh", "mean"),
    "medyan_kwh": ("kwh", "median"),
    "standart_sapma_kwh": ("kwh", "std"),
    "minimum_kwh": ("kwh", "min"),
    "maksimum_kwh": ("kwh", "max")
}

if "sozlesme_hesap_no" in df.columns:
    ilce_agg["benzersiz_musteri"] = ("sozlesme_hesap_no", "nunique")

ilce_ozet = (
    df.groupby("ilce", dropna=False)
    .agg(**ilce_agg)
    .reset_index()
    .sort_values("toplam_tuketim_kwh", ascending=False)
)

if "benzersiz_musteri" in ilce_ozet.columns:
    ilce_ozet["musteri_basina_toplam_tuketim_kwh"] = (
        ilce_ozet["toplam_tuketim_kwh"] /
        ilce_ozet["benzersiz_musteri"].replace(0, np.nan)
    )

display(ilce_ozet)

ilce_ozet.to_csv(
    os.path.join(CIKTI_KLASORU, "ilce_bazli_tanimlayici_istatistikler.csv"),
    sep=";",
    decimal=",",
    encoding="utf-8-sig",
    index=False
)


# ======================================================================
# 11. HESAP SINIFI BAZINDA TANIMLAYICI İSTATİSTİKLER
# ======================================================================

print("\n" + "=" * 80)
print("HESAP SINIFI BAZINDA TANIMLAYICI İSTATİSTİKLER")
print("=" * 80)

if "hesap_sinifi" in df.columns:
    hesap_agg = {
        "kayit_sayisi": ("kwh", "size"),
        "gecerli_kwh_sayisi": ("kwh", "count"),
        "toplam_kwh": ("kwh", "sum"),
        "ortalama_kwh": ("kwh", "mean"),
        "medyan_kwh": ("kwh", "median"),
        "standart_sapma_kwh": ("kwh", "std"),
        "minimum_kwh": ("kwh", "min"),
        "maksimum_kwh": ("kwh", "max")
    }

    if "sozlesme_hesap_no" in df.columns:
        hesap_agg["benzersiz_musteri"] = ("sozlesme_hesap_no", "nunique")

    hesap_sinifi_ozet = (
        df.groupby("hesap_sinifi", dropna=False)
        .agg(**hesap_agg)
        .reset_index()
        .sort_values("toplam_kwh", ascending=False)
    )

    display(hesap_sinifi_ozet.head(30))

    hesap_sinifi_ozet.to_csv(
        os.path.join(CIKTI_KLASORU, "hesap_sinifi_bazli_istatistikler.csv"),
        sep=";",
        decimal=",",
        encoding="utf-8-sig",
        index=False
    )
else:
    print("'hesap_sinifi' sütunu bulunamadı.")


# ======================================================================
# 12. AYKIRI DEĞER ANALİZİ
#     İlçe + hesap sınıfı bazında IQR
# ======================================================================

print("\n" + "=" * 80)
print("AYKIRI DEĞER ANALİZİ - GRUP BAZLI IQR")
print("=" * 80)

grup_sutunlari = [
    col for col in ["ilce", "hesap_sinifi"]
    if col in df.columns
]

df["aykiri_mi"] = False
df["iqr_alt_sinir"] = np.nan
df["iqr_ust_sinir"] = np.nan

if grup_sutunlari:
    def grup_iqr_hesapla(grup):
        grup = grup.copy()
        gecerli = grup["kwh"].dropna()

        # Çok küçük gruplarda güvenilir IQR hesaplaması yapılmaz
        if len(gecerli) < 4:
            grup["iqr_alt_sinir"] = np.nan
            grup["iqr_ust_sinir"] = np.nan
            grup["aykiri_mi"] = False
            return grup

        q1 = gecerli.quantile(0.25)
        q3 = gecerli.quantile(0.75)
        iqr = q3 - q1
        alt_sinir = q1 - 1.5 * iqr
        ust_sinir = q3 + 1.5 * iqr

        grup["iqr_alt_sinir"] = alt_sinir
        grup["iqr_ust_sinir"] = ust_sinir
        grup["aykiri_mi"] = (
            grup["kwh"].notna() &
            (
                (grup["kwh"] < alt_sinir) |
                (grup["kwh"] > ust_sinir)
            )
        )

        return grup

    df = (
        df.groupby(grup_sutunlari, dropna=False, group_keys=False)
        .apply(grup_iqr_hesapla)
        .reset_index(drop=True)
    )
else:
    print("İlçe ve hesap sınıfı sütunları bulunamadığı için global IQR kullanılacak.")

    q1 = df["kwh"].quantile(0.25)
    q3 = df["kwh"].quantile(0.75)
    iqr = q3 - q1
    alt_sinir = q1 - 1.5 * iqr
    ust_sinir = q3 + 1.5 * iqr

    df["iqr_alt_sinir"] = alt_sinir
    df["iqr_ust_sinir"] = ust_sinir
    df["aykiri_mi"] = (
        df["kwh"].notna() &
        (
            (df["kwh"] < alt_sinir) |
            (df["kwh"] > ust_sinir)
        )
    )

df_aykiri = df[df["aykiri_mi"]].copy()

print(f"Aykırı kayıt sayısı : {len(df_aykiri):,}")
print(f"Aykırı kayıt oranı  : {guvenli_oran(len(df_aykiri), len(df)):.2f}%")

print(
    "\nYorum: Aykırı kayıtlar otomatik olarak silinmemiştir. "
    "Özellikle sanayi, tarım, kamu ve yüksek tüketimli ticari müşteriler için "
    "yüksek kWh değerleri gerçek iş davranışını temsil edebilir."
)

if not df_aykiri.empty:
    aykiri_ozet_grup = (
        df_aykiri.groupby(grup_sutunlari, dropna=False)
        .agg(
            aykiri_kayit_sayisi=("kwh", "size"),
            toplam_aykiri_kwh=("kwh", "sum"),
            ortalama_aykiri_kwh=("kwh", "mean"),
            medyan_aykiri_kwh=("kwh", "median"),
            maksimum_aykiri_kwh=("kwh", "max")
        )
        .reset_index()
        .sort_values("aykiri_kayit_sayisi", ascending=False)
    )

    display(aykiri_ozet_grup.head(30))

    aykiri_ozet_grup.to_csv(
        os.path.join(CIKTI_KLASORU, "aykiri_deger_grup_ozeti.csv"),
        sep=";",
        decimal=",",
        encoding="utf-8-sig",
        index=False
    )




İLÇE BAZINDA TANIMLAYICI İSTATİSTİKLER


,ilce,kayit_sayisi,gecerli_kwh_sayisi,toplam_tuketim_kwh,ortalama_kwh,medyan_kwh,standart_sapma_kwh,minimum_kwh,maksimum_kwh,benzersiz_musteri,musteri_basina_toplam_tuketim_kwh
1,GÜMÜŞHACIKÖY,765657,765657,"74,526,473.58",97.34,48.31,"1,077.76","-25,370.64","153,575.73",18190,"4,097.11"
0,GÖYNÜCEK,295223,295223,"26,472,614.19",89.67,45.09,742.28,"-4,208.64","105,687.69",7128,"3,713.89"
2,HAMAMÖZÜ,124818,124818,"8,846,428.21",70.87,40.56,389.22,"-1,242.99","25,941.60",2981,"2,967.60"



HESAP SINIFI BAZINDA TANIMLAYICI İSTATİSTİKLER


,hesap_sinifi,kayit_sayisi,gecerli_kwh_sayisi,toplam_kwh,ortalama_kwh,medyan_kwh,standart_sapma_kwh,minimum_kwh,maksimum_kwh,benzersiz_musteri
15,Mesken,1026609,1026609,"57,069,204.28",55.59,47.36,73.90,"-2,836.64","14,940.93",24450
27,Ticari Faaliyet - Yazıhane,91298,91298,"15,322,356.45",167.83,41.60,781.76,"-3,861.20","60,449.14",2146
26,Tarımsal Faaliyetler (Şahıs),17266,17266,"9,351,411.48",541.61,16.15,"3,627.96","-9,069.13","69,307.54",437
25,Tarımsal Faaliyetler (Kooperatif),1632,1632,"6,167,802.15","3,779.29",282.72,"9,426.94","-12,574.78","105,687.69",56
0,1 SAYILI CETVELDE YER ALAN KAMU İDARESİ,8500,8500,"5,851,753.58",688.44,23.86,"3,911.91",0.00,"76,153.92",249
13,Lisansız Üreticiler,180,180,"2,907,945.80","16,155.25",322.32,"35,926.42",0.00,"153,575.73",7
33,İçme-Kullanma Suyu (Belediye),417,417,"2,174,033.37","5,213.51",400.02,"8,948.07",0.00,"39,479.28",10
12,Köy İçme Suyu Temini ve Dağıtımı Tesisi,2460,2460,"1,665,260.85",676.94,208.66,"1,212.33",0.00,"12,250.74",70
23,Sanayi,187,187,"1,363,940.53","7,293.80","1,531.68","13,716.34",0.00,"109,546.29",10
3,Belediye,2097,2097,"1,259,418.85",600.58,78.27,"3,350.98","-25,370.64","50,741.28",53



AYKIRI DEĞER ANALİZİ - GRUP BAZLI IQR
Aykırı kayıt sayısı : 42,279
Aykırı kayıt oranı  : 3.57%

Yorum: Aykırı kayıtlar otomatik olarak silinmemiştir. Özellikle sanayi, tarım, kamu ve yüksek tüketimli ticari müşteriler için yüksek kWh değerleri gerçek iş davranışını temsil edebilir.


,ilce,hesap_sinifi,aykiri_kayit_sayisi,toplam_aykiri_kwh,ortalama_aykiri_kwh,medyan_aykiri_kwh,maksimum_aykiri_kwh
37,GÜMÜŞHACIKÖY,Mesken,14991,"3,636,539.81",242.58,205.04,"3,208.08"
48,GÜMÜŞHACIKÖY,Ticari Faaliyet - Yazıhane,7229,"7,552,977.64","1,044.82",500.87,"60,449.14"
8,GÖYNÜCEK,Mesken,6959,"1,898,662.54",272.84,190.38,"14,940.93"
65,HAMAMÖZÜ,Mesken,2896,"588,308.15",203.15,175.29,"2,596.59"
17,GÖYNÜCEK,Ticari Faaliyet - Yazıhane,2421,"2,166,014.17",894.68,597.52,"6,304.56"
47,GÜMÜŞHACIKÖY,Tarımsal Faaliyetler (Şahıs),1614,"6,742,755.89","4,177.67",282.72,"69,307.54"
75,HAMAMÖZÜ,Ticari Faaliyet - Yazıhane,907,"1,206,570.25","1,330.29",603.67,"21,528.36"
26,GÜMÜŞHACIKÖY,1 SAYILI CETVELDE YER ALAN KAMU İDARESİ,713,"4,062,770.04","5,698.13","1,565.04","76,153.92"
55,GÜMÜŞHACIKÖY,Şantiye ve Geçici Aboneler,436,"366,588.80",840.80,309.41,"17,236.04"
16,GÖYNÜCEK,Tarımsal Faaliyetler (Şahıs),378,"2,161,311.02","5,717.75","1,904.81","45,289.93"


In [7]:

# ======================================================================
# 13. MÜŞTERİ BAZINDA TÜKETİM ÖZETİ
# ======================================================================

print("\n" + "=" * 80)
print("MÜŞTERİ BAZINDA TÜKETİM ÖZETİ")
print("=" * 80)

if "sozlesme_hesap_no" in df.columns:
    musteri_agg = {
        "toplam_tuketim_kwh": ("kwh", "sum"),
        "ortalama_tuketim_kwh": ("kwh", "mean"),
        "medyan_tuketim_kwh": ("kwh", "median"),
        "standart_sapma_kwh": ("kwh", "std"),
        "minimum_tuketim_kwh": ("kwh", "min"),
        "maksimum_tuketim_kwh": ("kwh", "max"),
        "fatura_kayit_sayisi": ("kwh", "count"),
        "ilce": ("ilce", "first")
    }

    if "hesap_sinifi" in df.columns:
        musteri_agg["hesap_sinifi"] = ("hesap_sinifi", mod_deger)

    if "mali_yil_donem" in df.columns:
        musteri_agg["aktif_donem_sayisi"] = ("mali_yil_donem", "nunique")
        musteri_agg["ilk_donem"] = ("mali_yil_donem", "min")
        musteri_agg["son_donem"] = ("mali_yil_donem", "max")
    elif "fatura_tarihi" in df.columns:
        musteri_agg["aktif_donem_sayisi"] = ("fatura_tarihi", "nunique")
        musteri_agg["ilk_donem"] = ("fatura_tarihi", "min")
        musteri_agg["son_donem"] = ("fatura_tarihi", "max")

    musteri_ozet = (
        df.groupby("sozlesme_hesap_no", dropna=False)
        .agg(**musteri_agg)
        .reset_index()
    )

    # Toplam tüketimde yüzdelik dilimler
    musteri_ozet["toplam_tuketim_yuzdelik"] = (
        musteri_ozet["toplam_tuketim_kwh"]
        .rank(pct=True, method="average") * 100
    )

    # "VIP" yerine kanıta dayalı ve nötr etiket
    musteri_ozet["tuketim_segmenti"] = pd.cut(
        musteri_ozet["toplam_tuketim_yuzdelik"],
        bins=[0, 50, 80, 95, 100],
        labels=[
            "Düşük tüketim",
            "Orta tüketim",
            "Yüksek tüketim",
            "En yüksek %5"
        ],
        include_lowest=True
    )

    yuksek_tuketimli_musteriler = (
        musteri_ozet[
            musteri_ozet["toplam_tuketim_yuzdelik"] >= 95
        ]
        .sort_values("toplam_tuketim_kwh", ascending=False)
        .copy()
    )

    print(f"Toplam müşteri özeti sayısı       : {len(musteri_ozet):,}")
    print(f"En yüksek %5 tüketim segmenti     : {len(yuksek_tuketimli_musteriler):,}")

    display(musteri_ozet.sort_values(
        "toplam_tuketim_kwh",
        ascending=False
    ).head(20))

    musteri_ozet.to_csv(
        os.path.join(CIKTI_KLASORU, "musteri_bazli_tuketim_ozeti.csv"),
        sep=";",
        decimal=",",
        encoding="utf-8-sig",
        index=False
    )

    yuksek_tuketimli_musteriler.to_csv(
        os.path.join(CIKTI_KLASORU, "en_yuksek_yuzde_5_tuketimli_musteriler.csv"),
        sep=";",
        decimal=",",
        encoding="utf-8-sig",
        index=False
    )

    print(
        "\nNot: Bu segment yalnızca tüketim davranışını gösterir. "
        "Müşterinin finansal değeri veya VIP statüsü için tahakkuk tutarı ve ödeme düzeni de analiz edilmelidir."
    )
else:
    print("'sozlesme_hesap_no' sütunu bulunamadığı için müşteri bazlı özet oluşturulamadı.")
    musteri_ozet = pd.DataFrame()
    yuksek_tuketimli_musteriler = pd.DataFrame()





MÜŞTERİ BAZINDA TÜKETİM ÖZETİ
Toplam müşteri özeti sayısı       : 28,290
En yüksek %5 tüketim segmenti     : 1,415


,sozlesme_hesap_no,toplam_tuketim_kwh,ortalama_tuketim_kwh,medyan_tuketim_kwh,standart_sapma_kwh,minimum_tuketim_kwh,maksimum_tuketim_kwh,fatura_kayit_sayisi,ilce,hesap_sinifi,aktif_donem_sayisi,ilk_donem,son_donem,toplam_tuketim_yuzdelik,tuketim_segmenti
17108,6414845714,"2,634,973.74","44,660.57",0.00,"51,696.64",0.00,"153,575.73",59,GÜMÜŞHACIKÖY,Lisansız Üreticiler,29,2023-03-01,2025-07-01,100.00,En yüksek %5
20635,7553584057,"1,636,590.30","45,460.84","51,522.30","23,883.34",0.00,"76,153.92",36,GÜMÜŞHACIKÖY,1 SAYILI CETVELDE YER ALAN KAMU İDARESİ,31,2023-01-01,2025-07-01,100.00,En yüksek %5
23307,8398443615,"1,238,340.68","30,203.43","36,810.90","16,963.57",0.00,"52,719.98",41,GÜMÜŞHACIKÖY,Karayolları Genel Müdürlüğü Aydınlatma,31,2023-01-01,2025-07-01,99.99,En yüksek %5
11914,4777252213,"857,108.70","17,492.01","25,240.32","13,575.94",0.00,"31,561.74",49,GÜMÜŞHACIKÖY,1 SAYILI CETVELDE YER ALAN KAMU İDARESİ,31,2023-01-01,2025-07-01,99.99,En yüksek %5
8857,3798287663,"782,153.22","14,757.61","17,889.76","15,341.52","-25,370.64","50,741.28",53,GÜMÜŞHACIKÖY,Belediye,31,2023-01-01,2025-07-01,99.99,En yüksek %5
2027,16419436,"776,755.31","18,945.25","17,031.58","15,161.86",0.00,"60,449.14",41,GÜMÜŞHACIKÖY,Ticari Faaliyet - Yazıhane,31,2023-01-01,2025-07-01,99.98,En yüksek %5
12091,4831215580,"771,460.44","15,126.68","15,694.56","13,697.70",0.00,"37,104.28",51,GÜMÜŞHACIKÖY,İçme-Kullanma Suyu (Belediye),31,2023-01-01,2025-07-01,99.98,En yüksek %5
14286,5515560446,"566,836.64","15,319.91","14,603.60","12,553.69",0.00,"39,479.28",37,GÖYNÜCEK,İçme-Kullanma Suyu (Belediye),31,2023-01-01,2025-07-01,99.98,En yüksek %5
18741,6954808809,"504,251.61","8,846.52",380.64,"16,950.72",0.00,"69,307.54",57,GÜMÜŞHACIKÖY,Tarımsal Faaliyetler (Şahıs),31,2023-01-01,2025-07-01,99.97,En yüksek %5
15440,5890799009,"477,336.42","14,464.74",580.32,"27,110.28",0.00,"105,687.69",33,GÖYNÜCEK,Tarımsal Faaliyetler (Kooperatif),25,2023-01-01,2025-01-01,99.97,En yüksek %5



Not: Bu segment yalnızca tüketim davranışını gösterir. Müşterinin finansal değeri veya VIP statüsü için tahakkuk tutarı ve ödeme düzeni de analiz edilmelidir.


In [8]:

# ======================================================================
# 14. ANALİZ İÇİN TEMİZ VE İŞARETLENMİŞ VERİ SETLERİ
# ======================================================================

print("\n" + "=" * 80)
print("VERİ SETLERİNİN HAZIRLANMASI")
print("=" * 80)

# Ana analiz seti: kWh değeri mevcut ve negatif olmayan kayıtlar
# Sıfır tüketimler korunur; çünkü pasif/boş abonelik davranışını gösterebilir.
df_temiz = df[
    df["kwh"].notna() &
    (df["kwh"] >= 0)
].copy()

# Pozitif tüketim seti: dağılım, ortalama ve mevsimsellik analizlerinde kullanılabilir
df_pozitif_temiz = df[
    df["kwh"].notna() &
    (df["kwh"] > 0)
].copy()

# Aykırı değerler silinmeden işaretlenmiş tam set
df_isaretli = df.copy()

print(f"Orijinal tahakkuk kayıt sayısı      : {len(df):,}")
print(f"Negatif olmayan temiz kayıt sayısı  : {len(df_temiz):,}")
print(f"Pozitif tüketimli kayıt sayısı       : {len(df_pozitif_temiz):,}")
print(f"Negatif tüketimli kayıt sayısı       : {len(df_negatif):,}")
print(f"Eksik kWh kayıt sayısı               : {len(df_kwh_eksik):,}")
print(f"Aykırı olarak işaretlenen kayıt      : {len(df_aykiri):,}")

print(
    "\nTemizlik yaklaşımı: Eksik kWh ve negatif kWh kayıtları ana tüketim analiz setinden "
    "ayrılmıştır. Sıfır tüketimler korunmuştur. Aykırı değerler silinmemiş, "
    "'aykiri_mi' sütunuyla işaretlenmiştir."
)




VERİ SETLERİNİN HAZIRLANMASI
Orijinal tahakkuk kayıt sayısı      : 1,185,698
Negatif olmayan temiz kayıt sayısı  : 1,185,547
Pozitif tüketimli kayıt sayısı       : 1,130,170
Negatif tüketimli kayıt sayısı       : 151
Eksik kWh kayıt sayısı               : 0
Aykırı olarak işaretlenen kayıt      : 42,279

Temizlik yaklaşımı: Eksik kWh ve negatif kWh kayıtları ana tüketim analiz setinden ayrılmıştır. Sıfır tüketimler korunmuştur. Aykırı değerler silinmemiş, 'aykiri_mi' sütunuyla işaretlenmiştir.


In [9]:

# ======================================================================
# 15. DOSYALARI DIŞARI AKTAR
# ======================================================================

print("\n" + "=" * 80)
print("DOSYALAR DIŞARI AKTARILIYOR")
print("=" * 80)

ciktilar = {
    "tahakkuk_isaretli_tam_veri.csv": df_isaretli,
    "temizlenmis_veri.csv": df_temiz,
    "pozitif_tuketim_verisi.csv": df_pozitif_temiz,
    "negatif_tuketimler.csv": df_negatif,
    "sifir_tuketimler.csv": df_sifir,
    "eksik_kwh_kayitlari.csv": df_kwh_eksik,
    "aykiri_tuketim_kayitlari.csv": df_aykiri
}

if not musteri_ozet.empty:
    ciktilar["musteri_bazli_tuketim_ozeti.csv"] = musteri_ozet

if not yuksek_tuketimli_musteriler.empty:
    ciktilar["en_yuksek_yuzde_5_tuketimli_musteriler.csv"] = (
        yuksek_tuketimli_musteriler
    )

for dosya_adi, veri in ciktilar.items():
    yol = os.path.join(CIKTI_KLASORU, dosya_adi)

    veri.to_csv(
        yol,
        sep=";",
        decimal=",",
        encoding="utf-8-sig",
        index=False
    )

    print(f" {dosya_adi:<45} {len(veri):>12,} satır")


# ======================================================================
# 16. ÇOK SEKMELİ EXCEL ÖZET RAPORU
# ======================================================================

excel_yolu = os.path.join(
    CIKTI_KLASORU,
    "notebook_01_veri_kalitesi_ozet_raporu.xlsx"
)

try:
    with pd.ExcelWriter(excel_yolu, engine="openpyxl") as writer:
        genel_ozet.to_excel(
            writer,
            sheet_name="Genel_Ozet",
            index=False
        )

        ilce_ozet.to_excel(
            writer,
            sheet_name="Ilce_Ozet",
            index=False
        )

        if "hesap_sinifi_ozet" in locals():
            hesap_sinifi_ozet.to_excel(
                writer,
                sheet_name="Hesap_Sinifi_Ozet",
                index=False
            )

        tuketim_durum_ozeti.to_excel(
            writer,
            sheet_name="Tuketim_Durumlari",
            index=False
        )

        if not musteri_ozet.empty:
            musteri_ozet.sort_values(
                "toplam_tuketim_kwh",
                ascending=False
            ).head(1000).to_excel(
                writer,
                sheet_name="Musteri_Ozet_Ilk1000",
                index=False
            )

        if not df_aykiri.empty:
            df_aykiri.head(1000).to_excel(
                writer,
                sheet_name="Aykiri_Ilk1000",
                index=False
            )

    print(f"\n Excel özet raporu oluşturuldu: {excel_yolu}")

except ImportError:
    print(
        "\n[UYARI] openpyxl bulunamadığı için Excel raporu oluşturulamadı. "
        "Kurulum için: pip install openpyxl"
    )
except Exception as hata:
    print(f"\n[UYARI] Excel raporu oluşturulamadı: {hata}")


# ======================================================================
# 17. SONUÇ
# ======================================================================

print("\n" + "=" * 80)
print("NOTEBOOK 01 BAŞARIYLA TAMAMLANDI")
print("=" * 80)
print(f"Çıktı klasörü: {CIKTI_KLASORU}")
print(
    "Üretilen çıktılar; temiz veri, pozitif tüketim verisi, negatif ve sıfır "
    "tüketimler, aykırı kayıtlar, müşteri özetleri ve veri kalitesi tablolarını içerir."
)


DOSYALAR DIŞARI AKTARILIYOR
✓ tahakkuk_isaretli_tam_veri.csv                   1,185,698 satır
✓ temizlenmis_veri.csv                             1,185,547 satır
✓ pozitif_tuketim_verisi.csv                       1,130,170 satır
✓ negatif_tuketimler.csv                                 151 satır
✓ sifir_tuketimler.csv                                55,377 satır
✓ eksik_kwh_kayitlari.csv                                  0 satır
✓ aykiri_tuketim_kayitlari.csv                        42,279 satır
✓ musteri_bazli_tuketim_ozeti.csv                     28,290 satır
✓ en_yuksek_yuzde_5_tuketimli_musteriler.csv           1,415 satır

✓ Excel özet raporu oluşturuldu: C:\Users\ÇaY GüZeLi\Downloads\notebook_01_ciktilari\notebook_01_veri_kalitesi_ozet_raporu.xlsx

NOTEBOOK 01 BAŞARIYLA TAMAMLANDI
Çıktı klasörü: C:\Users\ÇaY GüZeLi\Downloads\notebook_01_ciktilari
Üretilen çıktılar; temiz veri, pozitif tüketim verisi, negatif ve sıfır tüketimler, aykırı kayıtlar, müşteri özetleri ve veri kalitesi tab